# Day 4: Training the MNIST digit classifier

Day 3 built the network (784 → 128 → ReLU → 64 → ReLU → 10) and the loss function.
Today it learns. This notebook covers:

1. **Backpropagation**: how each weight's gradient is found
2. **Optimizer**: how the gradients are used to update the weights (Adam)
3. **Training cycle**: the five steps repeated for every batch
4. **Epochs**: repeating the whole dataset several times
5. **Evaluation**: measuring accuracy on images the model has never seen

## 1. Setup and data (same as Day 3)

In [ ]:
import copy

import torch
from torch import nn, optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

torch.manual_seed(42)  # same random start every run -> reproducible results
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

BATCH_SIZE = 64
tf = transforms.ToTensor()  # pixels 0-255 -> 0-1

train_data = datasets.MNIST("data", train=True, download=True, transform=tf)
test_data = datasets.MNIST("data", train=False, download=True, transform=tf)

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE)  # never used for learning

print(f"train: {len(train_data):,} images -> {len(train_loader)} batches of {BATCH_SIZE}")
print(f"test : {len(test_data):,} images")

## 2. The model (same as Day 3)

In [ ]:
class DigitNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Flatten(),          # (B, 1, 28, 28) -> (B, 784)
            nn.Linear(784, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 10),     # one raw score (logit) per digit 0-9
        )

    def forward(self, x):
        return self.layers(x)


model = DigitNet().to(device)
print(model)
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")

## 3. Loss function and optimizer

* `CrossEntropyLoss` measures how wrong the model is (softmax is built in, so the model outputs raw scores).
* `Adam` decides how to change the weights. `model.parameters()` tells it which weights to update,
  and `lr` (learning rate) is the step size.

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

## 4. Backpropagation on one batch

`loss.backward()` walks backwards from the loss through every layer and stores each
weight's **gradient** in `weight.grad`: how much, and in which direction, the loss changes
if that weight is nudged up a little. Nothing is updated yet.

In [ ]:
images, labels = next(iter(train_loader))
images, labels = images.to(device), labels.to(device)

loss = loss_fn(model(images), labels)
print(f"loss = {loss.item():.4f}  (random guessing over 10 digits is about 2.303)")

optimizer.zero_grad()
loss.backward()

for name, p in model.named_parameters():
    print(f"{name:<14} shape {str(tuple(p.shape)):<12} gradient norm = {p.grad.norm().item():.4f}")

## 5. Gradient descent by hand

The simplest optimizer rule is: **new weight = old weight − lr × gradient**.
Here it is applied to a *copy* of the model, so the real model is untouched.
After one step, the loss on the same batch should go down.

In [ ]:
demo = copy.deepcopy(model)
lr = 0.1

before = loss_fn(demo(images), labels)
demo.zero_grad()
before.backward()

with torch.no_grad():
    for p in demo.parameters():
        p -= lr * p.grad  # step downhill

after = loss_fn(demo(images), labels)
print(f"loss before step: {before.item():.4f}")
print(f"loss after step : {after.item():.4f}")

## 6. The training cycle

For every batch, five steps, always in this order:

1. `optimizer.zero_grad()`: clear the previous batch's gradients (PyTorch adds new ones on top of old ones)
2. forward pass: images → scores
3. loss: scores and true labels → one number
4. `loss.backward()`: backpropagation fills in the gradients
5. `optimizer.step()`: Adam updates the weights

In [ ]:
def train_one_epoch(model, loader, loss_fn, optimizer):
    model.train()
    total_loss = 0.0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()             # 1. clear old gradients
        scores = model(images)            # 2. forward pass
        loss = loss_fn(scores, labels)    # 3. loss
        loss.backward()                   # 4. backpropagation
        optimizer.step()                  # 5. update the weights

        total_loss += loss.item() * images.size(0)
    return total_loss / len(loader.dataset)  # average loss over the whole epoch

## 7. Evaluation

On the test set we only *measure*, we do not learn:

* `model.eval()` switches the model to evaluation mode
* `torch.no_grad()` turns off gradient tracking, which saves memory and time
* `argmax` picks the digit with the highest score, and accuracy is correct predictions ÷ total images

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        predictions = model(images).argmax(dim=1)
        correct += (predictions == labels).sum().item()
    return correct / len(loader.dataset)

## 8. Train for 5 epochs

One **epoch** is one full pass over all 60,000 training images (938 batches, so 938 updates).
Five epochs means 5 × 938 = 4,690 updates in total.

In [ ]:
EPOCHS = 5

# fresh model and optimizer, so the demos above don't affect training
model = DigitNet().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)

print(f"untrained test accuracy: {evaluate(model, test_loader):.2%}")
print(f"total updates: {EPOCHS * len(train_loader):,}\n")

history = []
for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader, loss_fn, optimizer)
    test_acc = evaluate(model, test_loader)
    history.append((train_loss, test_acc))
    print(f"epoch {epoch}/{EPOCHS}   train loss {train_loss:.4f}   test accuracy {test_acc:.2%}")

## 9. Final result and a single prediction

In [ ]:
print(f"final test accuracy: {evaluate(model, test_loader):.2%}")

image, label = test_data[0]
model.eval()
with torch.no_grad():
    probs = torch.softmax(model(image.unsqueeze(0).to(device)), dim=1)[0]

print(f"\ntrue digit: {label}   predicted: {probs.argmax().item()}")
for digit, p in enumerate(probs.tolist()):
    print(f"  {digit}: {p:6.2%} " + "#" * round(p * 40))